In [ ]:
# ── Starter Cell 1: Load environment variables ──────────────────────────────
from dotenv import load_dotenv   # load_dotenv reads .env file into os.environ
import os                         # os.getenv safely retrieves env vars

load_dotenv()                     # parse .env in current working directory
api_key = os.getenv("ANTHROPIC_API_KEY")  # fetch the key; None if missing
print("API key loaded:", api_key is not None)  # confirm without printing secret

In [ ]:
# ── Starter Cell 2: Initialise Anthropic client ──────────────────────────────
from anthropic import Anthropic   # main SDK class that wraps all API calls

client = Anthropic(api_key=api_key)  # create one shared client for the session
print("Anthropic client ready")       # quick confirmation that init succeeded

In [ ]:
# ── Starter Cell 3: Smoke test ───────────────────────────────────────────────
response = client.messages.create(
    model="claude-haiku-4-5",         # fast/cheap model ideal for tests
    # model="claude-sonnet-4-5",      # swap here for higher quality
    max_tokens=50,                    # tiny limit — just checking connectivity
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)
print(response.content[0].text)       # content is a list; [0] is the first block

# CCA Lab: Response Streaming

**Course:** Building with the Claude API  
**Sub-module:** Accessing Claude with the API  
**Lesson:** Response Streaming

## What this lab covers
- Why streaming dramatically improves perceived latency in chat applications
- The six stream-event types Claude emits and which carry user-visible text
- Raw event iteration (`stream=True`) vs. the SDK's high-level `text_stream`
- Retrieving the fully-assembled `Message` object after streaming completes
- Failure modes unique to streaming (missing `max_tokens`, interrupted streams)
- An improvement cycle that scores and refines streamed prompt quality
- Per-call token tracking across every API call in the notebook

## CCA domains exercised
- **Primary:** Agentic Architecture

## Learning objectives
1. Explain the six streaming event types and identify which carries generated text.
2. Implement a live streaming call using both the raw and simplified SDK interfaces.
3. Retrieve `get_final_message()` and extract token usage after stream completion.
4. Apply an improvement cycle with a rubric to score and revise a streaming prompt.
5. Diagnose at least three streaming failure modes from symptom to root cause.

## Section 1: Prerequisites, Environment Setup & API Glossary
### CCA objective demonstrated: Configure the development environment and interpret every parameter available in a streaming Messages API call

Before writing streaming code, understand every knob you can turn.

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `model` | string | ✅ | Model ID, e.g. `claude-haiku-4-5` |
| `max_tokens` | int | ✅ | Hard ceiling on output tokens — **must not be omitted** |
| `messages` | list[dict] | ✅ | Alternating `user`/`assistant` turns |
| `system` | string | ❌ | Persistent instructions applied before every turn |
| `stream` | bool | ❌ | `True` → raw Server-Sent Events; default `False` |
| `temperature` | float 0–1 | ❌ | Sampling randomness; 0 = deterministic |
| `top_p` | float 0–1 | ❌ | Nucleus sampling; alternative to temperature |
| `stop_sequences` | list[str] | ❌ | Halt generation when any sequence is produced |

### Streaming-specific SDK interfaces

| Interface | Returns | Best for |
|-----------|---------|----------|
| `client.messages.create(stream=True)` | Raw event iterator | Inspecting event types, custom parsers |
| `client.messages.stream(...)` context manager | `StreamContext` with `.text_stream` | Production chat UIs, cleaner code |
| `stream.get_final_message()` | Complete `Message` object | Storage, token accounting, chaining |

## Section 2: ask() Helper with Statement-by-Statement Annotation
### CCA objective demonstrated: Build a reusable, session-aware helper that centralises token tracking and supports both streaming and standard calls

In [ ]:
# ── ask() helper — fully annotated ──────────────────────────────────────────

usage_log = []   # module-level list; every API call appends a dict here
                 # persists across cells so we can report session totals later

def ask(
    prompt,                          # str: the user message text
    system="",                       # str: optional system instructions
    model="claude-haiku-4-5",        # str: model ID — Haiku is fastest/cheapest
    max_tokens=256,                  # int: REQUIRED — caps output to prevent runaway cost
    temperature=0,                   # float: 0 = fully deterministic, ideal for evals
):
    """
    Send a single-turn message to Claude and return the response text.

    Parameters
    ----------
    prompt      : User message content.
    system      : System prompt; injected as a top-level keyword argument,
                  NOT as a messages entry, because the API reserves the
                  'system' key specifically for persistent instructions.
    model       : Claude model identifier string.
    max_tokens  : Hard output-token ceiling; the API rejects calls without it.
    temperature : Sampling temperature; 0 gives reproducible outputs.

    Returns
    -------
    str — the assistant's reply text.
    """
    kwargs = {                        # build params dict incrementally so
        "model": model,               #   we can conditionally add 'system'
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}],
                                      # messages is a list of dicts;
                                      # single-turn = one user entry
    }
    if system:                        # only add 'system' when non-empty;
        kwargs["system"] = system     #   passing "" would be needless noise

    resp = client.messages.create(**kwargs)
                                      # **kwargs unpacks dict as keyword args;
                                      # cleaner than a long positional list

    usage_log.append({                # record usage AFTER the call succeeds
        "call": len(usage_log) + 1,   # human-readable 1-based call number
        "label": prompt[:40],         # first 40 chars identify the call
        "input_tokens": resp.usage.input_tokens,   # tokens Claude read
        "output_tokens": resp.usage.output_tokens, # tokens Claude generated
    })

    # resp.content is a list because a message CAN contain multiple blocks
    # (e.g. a text block followed by a tool_use block).  We access [0] because
    # for a plain text response there is exactly one TextBlock at index 0.
    # resp.stop_reason tells us WHY generation ended:
    #   'end_turn'    — model chose to stop (normal)
    #   'max_tokens'  — hit the ceiling we set (response may be truncated!)
    #   'stop_sequence' — a stop_sequence was triggered
    return resp.content[0].text       # return just the string for convenience

# Quick sanity check
print(ask("What is response streaming in one sentence?"))

## Section 3: System Prompt — What It Is, When & Why
### CCA objective demonstrated: Distinguish system-level instructions from user-turn content and apply the correct placement rule in streaming contexts

The `system` parameter lets you bake persistent, reusable instructions into every call without polluting the `messages` list.

| Put it in `system` | Put it in the `user` turn |
|---|---|
| Persona, tone, role definition | The actual question or task |
| Output format rules (JSON, markdown) | Dynamic context (user name, session data) |
| Safety / scope constraints | Examples specific to this request |
| Domain knowledge that never changes | Real-time data or retrieved documents |

**Architectural principle:** The `system` parameter encodes *what the model is*; the `messages` list encodes *what is happening right now*.

## Section 4: D-STREAM — Response Streaming: Events, Chunks, and Reassembly
### CCA objective demonstrated: Implement live streaming with both raw event iteration and the simplified text_stream interface; retrieve the final assembled Message object

Streaming works in two SDK flavours:

1. **`messages.create(stream=True)`** — returns a raw iterator of typed event objects.  
   Use this when you need to inspect every event type (e.g. for custom parsers).

2. **`messages.stream(...)` context manager** — wraps the raw stream and exposes `.text_stream`, which silently discards non-text events.  
   Use this for production chat UIs — it is cleaner and safer.

After either stream closes, `stream.get_final_message()` returns the fully-assembled `Message` object, giving you token counts and `stop_reason` for logging.

In [ ]:
# ── Part A: Raw event iteration ──────────────────────────────────────────────
# Goal: see ALL event types emitted by Claude during a single streaming call.

messages = [{"role": "user",          # minimal single-turn messages list
             "content": "Write a 1-sentence description of a fake database."}]

print("=== Raw streaming events ===")

raw_stream = client.messages.create(  # create() with stream=True returns an
    model="claude-haiku-4-5",          #   iterator rather than a complete Message
    max_tokens=120,                    # REQUIRED even for streaming calls
    messages=messages,
    stream=True,                       # <── this flag enables streaming
)

event_counts = {}                      # dict to tally how many of each type we see

for event in raw_stream:               # iterate; each event is a typed SDK object
    event_type = type(event).__name__  # e.g. 'RawMessageStartEvent'
    event_counts[event_type] = event_counts.get(event_type, 0) + 1
                                       # .get(k, 0) avoids KeyError on first occurrence

    # Only print ContentBlockDelta events so output isn't overwhelming
    # ContentBlockDelta carries the actual generated text chunks
    if "Delta" in event_type and hasattr(event, "delta"):
        delta = event.delta            # the delta object contains the text fragment
        if hasattr(delta, "text"):     # guard: tool_use deltas have no .text attr
            print(f"  [ContentBlockDelta] → {repr(delta.text)}")  # repr shows whitespace

print("\n=== Event type counts ===")
for etype, count in event_counts.items():  # summarise what fired
    print(f"  {etype}: {count}")

# Append to usage_log manually — raw stream doesn't go through ask()
usage_log.append({
    "call": len(usage_log) + 1,
    "label": "raw-stream demo",
    "input_tokens": 0,                 # raw stream doesn't expose usage mid-flight;
    "output_tokens": 0,                # we will capture usage in the text_stream demo
})
print("\nRaw stream complete.")

In [ ]:
# ── Part B: Simplified text_stream + get_final_message() ────────────────────
# This is the recommended production pattern for chat applications.

print("=== Simplified text_stream ===")
print("Streaming response: ", end="", flush=True)  # flush=True forces immediate display

with client.messages.stream(           # context manager automatically closes the
    model="claude-haiku-4-5",           #   HTTP connection when the block exits
    max_tokens=200,
    system="You are a witty senior DBA.",  # system prompt injected here
    messages=[
        {"role": "user",
         "content": "In 2-3 sentences, describe what makes a great database schema."}
    ],
) as stream:                           # 'stream' is a StreamContext object

    for text in stream.text_stream:    # .text_stream yields only text strings;
                                       #   non-text events are silently discarded
        print(text, end="", flush=True)  # end="" prevents newlines between chunks
                                         # flush=True ensures chunks appear immediately

    # After the for-loop the stream is fully consumed but the context is still open
    final_msg = stream.get_final_message()  # assembles all chunks into one Message obj

print("\n")  # newline after streamed output
print("=== Final message object ===")
print(f"  stop_reason  : {final_msg.stop_reason}")     # 'end_turn' = clean finish
print(f"  input_tokens : {final_msg.usage.input_tokens}")
print(f"  output_tokens: {final_msg.usage.output_tokens}")
print(f"  full_text    : {final_msg.content[0].text[:80]}...")  # first 80 chars

# Record this call in the session usage log
usage_log.append({
    "call": len(usage_log) + 1,
    "label": "text_stream demo",
    "input_tokens": final_msg.usage.input_tokens,
    "output_tokens": final_msg.usage.output_tokens,
})
print("Usage logged ✅")

## Section 5: Multi-Turn Conversation — Messages List Accumulation
### CCA objective demonstrated: Build a stateful conversation by manually accumulating the messages list across turns, printing its full state after each exchange

In [ ]:
# ── Multi-turn streaming conversation ────────────────────────────────────────
# We maintain the messages list by hand so it's transparent exactly what
# is sent to the API on each turn.

import textwrap  # for pretty-printing long strings at a fixed width

conv_messages = []   # start with an empty conversation

def stream_turn(user_text, messages_list, label=""):
    """
    Append a user message, stream Claude's reply, append the assistant reply,
    then print the full messages list so the reader can see it grow.

    Parameters
    ----------
    user_text     : The user's message string.
    messages_list : The running conversation list (mutated in-place).
    label         : Short string for the usage log.

    Returns
    -------
    str — the assistant reply text.
    """
    messages_list.append({"role": "user", "content": user_text})
                                               # Step 1: add user turn
    collected = []                             # accumulate text chunks here

    with client.messages.stream(
        model="claude-haiku-4-5",
        max_tokens=150,
        messages=messages_list,               # send full history every call
    ) as stream:
        for chunk in stream.text_stream:      # iterate text chunks
            collected.append(chunk)           # save each chunk
        final = stream.get_final_message()    # get assembled message

    reply_text = "".join(collected)           # join chunks into full reply
    messages_list.append({"role": "assistant", "content": reply_text})
                                               # Step 2: add assistant turn

    # Record usage
    usage_log.append({
        "call": len(usage_log) + 1,
        "label": label or user_text[:30],
        "input_tokens": final.usage.input_tokens,
        "output_tokens": final.usage.output_tokens,
    })

    # ── Print the FULL messages list after every turn ──────────────────────
    print(f"\n{'─'*60}")
    print(f"Messages list after turn {len(messages_list)//2}:")
    for i, msg in enumerate(messages_list):   # print every entry so reader sees growth
        role = msg['role'].upper()            # uppercase for readability
        content_preview = textwrap.shorten(msg['content'], width=70, placeholder="...")
                                               # shorten() truncates at word boundary
        print(f"  [{i}] {role}: {content_preview}")
    print(f"{'─'*60}")
    return reply_text

# Turn 1
stream_turn("What is a primary key?", conv_messages, label="multi-turn-t1")

# Turn 2 — references the previous answer
stream_turn("Can a table have more than one?", conv_messages, label="multi-turn-t2")

# Turn 3 — continues the thread
stream_turn("Give a one-sentence example.", conv_messages, label="multi-turn-t3")

## Section 6: D-ERR — Intentional Error Demonstration: Streaming Edge Cases
### CCA objective demonstrated: Trigger, catch, and explain the API errors most likely to occur in streaming code so you can recognise and fix them in production

In [ ]:
# ── Intentional error: omit max_tokens inside a streaming call ───────────────
# max_tokens is REQUIRED by the API. Omitting it raises a BadRequestError
# (HTTP 400) immediately — even in streaming mode.

from anthropic import BadRequestError   # specific exception for 4xx API errors

print("Attempting streaming call WITHOUT max_tokens...")

try:
    bad_stream = client.messages.create(
        model="claude-haiku-4-5",
        # max_tokens intentionally omitted ← this is the bug
        messages=[{"role": "user", "content": "Hello"}],
        stream=True,                   # streaming makes no difference; error fires
                                       # at request time, before any event arrives
    )
    for _ in bad_stream:               # iteration would start here if no error
        pass
except BadRequestError as e:
    print(f"\n✅ Caught BadRequestError (HTTP 400):")
    print(f"   {e}")
    print("\n👉 Fix: always pass max_tokens= to every messages.create() call.")
except Exception as e:
    print(f"\n⚠️  Unexpected error type {type(e).__name__}: {e}")

print("\nDemonstration complete — notebook continues safely.")

## Section 7: Improvement Cycle — Write → Evaluate → Improve → Re-evaluate
### CCA objective demonstrated: Apply a numeric rubric to score streaming prompts, iterate toward a passing threshold, and recognise the limits of keyword-based scoring

We will score prompts on three dimensions (10 pts each = 30 pts max) and iterate until the score reaches the passing threshold of 25.

> **Grader reliability note:** Deterministic rubrics can over-score responses that contain the right words but wrong semantics. For production evals, complement keyword checks with a Claude-as-judge semantic pass — the rule + judge layering pattern from the evaluation labs.

| Dimension | What we check | Points |
|-----------|--------------|--------|
| Clarity | Contains a clear action verb in the first sentence | 10 |
| Specificity | Mentions a concrete noun (database, schema, table, key) | 10 |
| Conciseness | Response ≤ 120 words | 10 |

In [ ]:
# ── Rubric scorer ────────────────────────────────────────────────────────────
import re   # regular expressions for pattern matching

def score_response(response_text):
    """
    Score a model response on three streaming-prompt quality dimensions.

    Parameters
    ----------
    response_text : str — the assistant reply to evaluate.

    Returns
    -------
    int — total score out of 30.
    """
    total = 0

    # ── Dimension 1: Clarity ─────────────────────────────────────────────────
    # re.search scans the WHOLE string (not just the start like re.match would);
    # we want to find the verb anywhere in the first sentence.
    # \b word boundaries prevent partial matches (e.g. 'describes' matching 'ribe').
    first_sentence = next(   # skip markdown headings to find first prose sentence
        (s.strip() for s in response_text.split("\n")
         if s.strip() and not s.strip().startswith("#")),
        ""
    )
    clarity_pattern = r"\b(explain|explains|describe|describes|define|defines|show|shows|demonstrate|demonstrates|outline|outlines|use|uses|identify|identifies|allow|allows|ensure|ensures)\b"
    clarity_hit = bool(re.search(clarity_pattern, first_sentence, re.IGNORECASE))
                                   # bool() converts Match object/None to True/False
    clarity_pts = int(clarity_hit) * 10   # int(True)=1, int(False)=0; multiply by weight
    icon = "✅" if clarity_hit else "❌"
    print(f"  {icon} Clarity      : {clarity_pts}/10 — action verb in first sentence")
    total += clarity_pts

    # ── Dimension 2: Specificity ─────────────────────────────────────────────
    # \b ensures 'table' doesn't match 'vegetable'; re.search checks full text.
    spec_pattern = r"\b(database|schema|table|key|index|query|column|row)\b"
    spec_hit = bool(re.search(spec_pattern, response_text, re.IGNORECASE))
    spec_pts = int(spec_hit) * 10
    icon = "✅" if spec_hit else "❌"
    print(f"  {icon} Specificity  : {spec_pts}/10 — concrete DB noun present")
    total += spec_pts

    # ── Dimension 3: Conciseness ─────────────────────────────────────────────
    # Split on whitespace; len() counts tokens (approximate, but consistent).
    word_count = len(response_text.split())
    concise_hit = word_count <= 120   # threshold: 120 words
    concise_pts = int(concise_hit) * 10
    icon = "✅" if concise_hit else "❌"
    print(f"  {icon} Conciseness  : {concise_pts}/10 — {word_count} words (limit 120)")
    total += concise_pts

    print(f"  {'─'*40}")
    print(f"  Total: {total}/30")
    return total

print("Scorer defined. Quick sanity test:")
_ = score_response("Describe the table structure. The database key is primary.")

In [ ]:
# ── Improvement cycle ────────────────────────────────────────────────────────
PASS_THRESHOLD = 25   # minimum score to consider the prompt 'production ready'

# Each version is a (prompt, system_prompt) tuple.
# V1 and V2 use no system prompt — baseline behaviour.
# V3 adds a system prompt that explicitly targets the failing rubric dimensions.
versions = [
    # Version 1 — intentionally vague; scores low on clarity + specificity
    ("Tell me about things in IT.", None),
    # Version 2 — improved prompt; adds domain focus but no system prompt
    ("Describe what a database schema is.", None),
    # Version 3 — optimised: system prompt forces action verb + no headings
    (
        "Define a database primary key in two sentences.",
        "Do not use markdown headings. Start your response directly with an "
        "action verb such as 'A primary key defines' or 'Use a primary key to'. "
        "Keep the answer under 60 words. Use the word 'primary key' explicitly."
    ),
]

results = []   # store per-version results for comparison table

for version, (prompt, system) in enumerate(versions, start=1):   # unpack tuple
    print(f"\n{chr(9552)*60}")
    print(f"Version {version}: {prompt}")
    if system:
        print(f"System : {system[:80]}...")
    print(f"{chr(9472)*60}")

    reply = ask(prompt, system=system, max_tokens=150)   # pass system when present

    print(f"Response: {reply[:120]}..." if len(reply) > 120 else f"Response: {reply}")
    print("Scores:")
    s = score_response(reply)             # prints per-dimension results

    results.append({
        "version": version,
        "prompt_trunc": prompt[:35],
        "score": s,
    })

# ── Side-by-side comparison table ────────────────────────────────────────────
print(f"\n{chr(9552)*60}")
print("COMPARISON TABLE")
print(f"{chr(9472)*60}")
print(f"{'Ver':<5} {'Prompt':<38} {'Score':>5}")
print(f"{'─'*5} {'─'*38} {'─'*5}")
for r in results:
    print(f"{r['version']:<5} {r['prompt_trunc']:<38} {r['score']:>5}/30")

best_score = max(r["score"] for r in results)
status = "PASS ✅" if best_score >= PASS_THRESHOLD else "FAIL ❌"
print(f"\nBest score: {best_score}/30  →  {status} (threshold {PASS_THRESHOLD}/30)")


## Section 8: Failure Mode Analysis
### CCA objective demonstrated: Identify, reproduce, and remediate the most common streaming failure modes to build production-resilient code

| Failure Mode | Trigger | Symptom | Fix |
|---|---|---|---|
| Missing `max_tokens` | `max_tokens` omitted from any `messages.create()` call | `BadRequestError` HTTP 400 immediately | Always pass `max_tokens=` |
| Truncated stream | `max_tokens` too small; model hits limit mid-sentence | Response ends abruptly; `stop_reason="max_tokens"` | Raise `max_tokens` or chunk large outputs |
| Iterating after context exit | Calling `.text_stream` outside the `with` block | `RuntimeError` or silent empty iterator | Keep all iteration inside `with ... as stream:` |
| Forgetting `flush=True` | Printing chunks without `flush=True` in a terminal | All text appears at once at stream end, not chunk-by-chunk | Add `flush=True` to every `print(text, end="")` |

The live demo below deliberately triggers the **truncated stream** failure and confirms `stop_reason="max_tokens"`.

In [ ]:
# ── Live failure demo: truncated stream (max_tokens too small) ───────────────
print("=== Failure Mode: Truncated Stream ===")
print("Requesting a long response with max_tokens=10 (way too small)...\n")

truncated_chunks = []   # collect chunks to show what actually arrived

with client.messages.stream(
    model="claude-haiku-4-5",
    max_tokens=10,                     # deliberately tiny — truncation guaranteed
    messages=[
        {"role": "user",
         "content": "Explain relational databases in 200 words."}
    ],
) as stream:
    for chunk in stream.text_stream:   # collect whatever arrives before cutoff
        truncated_chunks.append(chunk)
    truncated_final = stream.get_final_message()  # still accessible after stream ends

truncated_text = "".join(truncated_chunks)  # join outside loop for efficiency

print(f"Received text  : {repr(truncated_text)}")
print(f"stop_reason    : {truncated_final.stop_reason}")
print(f"output_tokens  : {truncated_final.usage.output_tokens}")

# ── Diagnose and prescribe fix ───────────────────────────────────────────────
if truncated_final.stop_reason == "max_tokens":   # programmatic check
    print("\n⚠️  DETECTED: stop_reason is 'max_tokens' — response was truncated!")
    print("   Diagnosis : max_tokens (10) was exhausted before the model finished.")
    print("   Fix       : Increase max_tokens to at least 300 for this request.")
else:
    print("\n✅ Clean stop — no truncation detected.")

# ── Log usage ────────────────────────────────────────────────────────────────
usage_log.append({
    "call": len(usage_log) + 1,
    "label": "failure-truncated-stream",
    "input_tokens": truncated_final.usage.input_tokens,
    "output_tokens": truncated_final.usage.output_tokens,
})

## Section 9: Token Usage Tracking Across the Session
### CCA objective demonstrated: Aggregate per-call token counts into a session report that supports cost estimation and usage optimisation

In [ ]:
# ── Session token usage report ───────────────────────────────────────────────
# Prints a per-call table then cumulative totals.

print("=== Per-Call Token Usage ===")
print(f"{'#':<5} {'Label':<35} {'Input':>8} {'Output':>8} {'Total':>8}")
print(f"{'─'*5} {'─'*35} {'─'*8} {'─'*8} {'─'*8}")

cum_input = 0    # running cumulative input token count
cum_output = 0   # running cumulative output token count

for entry in usage_log:              # iterate every recorded call
    inp = entry["input_tokens"]      # tokens the model read this call
    out = entry["output_tokens"]     # tokens the model generated this call
    call_total = inp + out           # per-call token total
    cum_input += inp                 # accumulate across calls
    cum_output += out

    label = entry['label']           # extract label to avoid backslash in f-string
    print(f"{entry['call']:<5} {label:<35} {inp:>8} {out:>8} {call_total:>8}")

print(f"{'─'*5} {'─'*35} {'─'*8} {'─'*8} {'─'*8}")
session_total = cum_input + cum_output  # grand total for the session
print(f"{'TOTAL':<5} {'':35} {cum_input:>8} {cum_output:>8} {session_total:>8}")

print(f"\n📊 Session summary")
print(f"   API calls made : {len(usage_log)}")
print(f"   Cumulative input  tokens: {cum_input:,}")
print(f"   Cumulative output tokens: {cum_output:,}")
print(f"   Grand total tokens      : {session_total:,}")

# Cost estimation (Haiku pricing as of 2024; verify current prices)
# Haiku: $0.25 / 1M input, $1.25 / 1M output
est_cost = (cum_input / 1_000_000 * 0.25) + (cum_output / 1_000_000 * 1.25)
print(f"   Estimated cost (Haiku)  : ${est_cost:.6f}")

## Section 10: Practice Drills
### CCA objective demonstrated: Apply streaming concepts independently through three scaffolded exercises that mirror real CCA exam scenarios

Complete each drill by replacing the `# YOUR CODE HERE` placeholders.

**Drill 1 — Event type census:** Stream a response and print a dictionary counting every distinct event type name emitted during the stream.  
**Drill 2 — Word-by-word display:** Stream a 3-sentence story. Print each word separately (split chunks on spaces) with a 0.05-second delay to simulate a typewriter effect.  
**Drill 3 — Stop-reason guard:** Write a helper `safe_stream(prompt)` that raises a descriptive `RuntimeError` if `get_final_message().stop_reason` is `"max_tokens"`, and returns the text otherwise.

In [ ]:
# ── Drill 1: Event type census ────────────────────────────────────────────────
# Stream any prompt and print a dict of {EventTypeName: count}.

# YOUR CODE HERE
# Hint: Use client.messages.create(stream=True) and type(event).__name__

# ── Reference solution (uncomment to check your work) ────────────────────────
# event_census = {}
# s = client.messages.create(
#     model="claude-haiku-4-5", max_tokens=80,
#     messages=[{"role": "user", "content": "Name three colours."}],
#     stream=True,
# )
# for ev in s:
#     k = type(ev).__name__
#     event_census[k] = event_census.get(k, 0) + 1
# print(event_census)

In [ ]:
# ── Drill 2: Word-by-word typewriter display ──────────────────────────────────
import time   # needed for sleep

# YOUR CODE HERE
# Hint: for each chunk in stream.text_stream, split on " ",
#       then print each word + " " with flush=True and time.sleep(0.05)

# ── Reference solution (uncomment to check) ───────────────────────────────────
# with client.messages.stream(
#     model="claude-haiku-4-5", max_tokens=120,
#     messages=[{"role": "user", "content": "Tell a 3-sentence story about a robot."}],
# ) as s:
#     for chunk in s.text_stream:
#         for word in chunk.split(" "):
#             if word:
#                 print(word, end=" ", flush=True)
#                 time.sleep(0.05)

In [ ]:
# ── Drill 3: Stop-reason guard ────────────────────────────────────────────────
# Write safe_stream(prompt) — raises RuntimeError on truncation, else returns text.

# YOUR CODE HERE

# ── Reference solution (uncomment to check) ───────────────────────────────────
# def safe_stream(prompt, max_tokens=200):
#     """Stream prompt; raise RuntimeError if response is truncated."""
#     with client.messages.stream(
#         model="claude-haiku-4-5", max_tokens=max_tokens,
#         messages=[{"role": "user", "content": prompt}],
#     ) as s:
#         text = "".join(s.text_stream)
#         final = s.get_final_message()
#     if final.stop_reason == "max_tokens":
#         raise RuntimeError(
#             f"Stream truncated at {max_tokens} tokens. Raise max_tokens and retry."
#         )
#     return text
#
# # Test the guard with a tiny limit
# try:
#     safe_stream("Explain quantum physics.", max_tokens=5)
# except RuntimeError as e:
#     print(f"Caught: {e}")

## Section 11: CCA Takeaways — 5 Exam-Ready Mental Models
### CCA objective demonstrated: Consolidate streaming knowledge into concise, exam-applicable principles

| # | Mental Model | Why It Matters for the CCA Exam |
|---|---|---|
| 1 | **`max_tokens` is mandatory — even for streaming** | Omitting it raises an immediate `BadRequestError`; this is the most common streaming bug in exam scenarios. |
| 2 | **`ContentBlockDelta` carries user-visible text; all other event types are control signals** | In agentic pipelines you only forward deltas to the client; swallowing other events is correct behaviour. |
| 3 | **`messages.stream()` context manager > `messages.create(stream=True)`** | The context manager handles connection cleanup, exposes `.text_stream`, and provides `get_final_message()` — prefer it in production. |
| 4 | **`get_final_message()` is the only way to access token usage after streaming** | Streaming does not return a `Message` object inline; you must call this method inside the `with` block after iteration completes. |
| 5 | **`stop_reason="max_tokens"` is a programmatic signal, not an error** | Detect it after `get_final_message()` to implement retry logic, chunked generation, or user-facing warnings in agentic systems. |